# Document Search with LangChain

## Set up the RAG workflow environment

#### Import libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import requests
import sys

from pathlib import Path

from langchain.chains import RetrievalQA
from langchain_community.vectorstores import FAISS
from langchain.document_loaders.pdf import PyPDFDirectoryLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter

#### Load config files

In [36]:
import os
from langchain.document_loaders import PyPDFLoader

In [12]:
!cd /../

In [17]:
os.chdir('/fs01/home/ws_ikharchuk/rag_bootcamp_ik/document_search')

In [18]:
# Add root folder of the rag_bootcamp repo to PYTHONPATH
current_dir = Path().resolve()
parent_dir = current_dir.parent
sys.path.insert(0, str(parent_dir))

from utils.load_secrets import load_env_file
load_env_file()

In [19]:
GENERATOR_BASE_URL = os.environ.get("OPENAI_BASE_URL")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

#### Set up some helper functions

In [20]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

#### Make sure other necessary items are in place

In [21]:
# Look for the source_documents folder and make sure there is at least 1 pdf file here
contains_pdf = False
directory_path = "./source_documents"
if not os.path.exists(directory_path):
    print(f"ERROR: The {directory_path} subfolder must exist under this notebook")
for filename in os.listdir(directory_path):
    contains_pdf = True if ".pdf" in filename else contains_pdf
if not contains_pdf:
    print(f"ERROR: The {directory_path} subfolder must contain at least one .pdf file")

#### Choose LLM and embedding model

In [22]:
GENERATOR_MODEL_NAME = "Meta-Llama-3.1-8B-Instruct"
#GENERATOR_MODEL_NAME = 'DeepSeek-R1-Distill-Qwen-1.5B'
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"


## Start with a basic generation request without RAG augmentation



## Create model instance 

In [23]:
llm = ChatOpenAI(
    model=GENERATOR_MODEL_NAME,
    temperature=0,
    max_tokens=None,
    base_url=GENERATOR_BASE_URL,
    api_key=OPENAI_API_KEY
)

## Ingestion: Load and store the documents from `source_documents`

Start by reading in all the PDF files from `source_documents`, break them up into smaller digestible chunks, then encode them as vector embeddings.

In [24]:

from langchain.document_loaders import TextLoader

In [25]:
ls /projects/RAG2/scotia-2/Datasets-Scotia-2/IBIS

'11114CA Wheat Farming in Canada Industry Report.pdf'
'11115CA Corn Farming in Canada Industry Report.pdf'
'33639CA Auto Parts Manufacturing in Canada Industry Report.pdf'
'44111CA New Car Dealers in Canada Industry Report.pdf'
'48412CA Long-Distance Freight Trucking in Canada Industry Report.pdf'
'48422CA Local Specialized Freight Trucking in Canada Industry Report.pdf'
'48423CA Long-Distance Specialized Freight Trucking in Canada Industry Report.pdf'


### Setup embeding models

In [26]:
model_kwargs = {'device': 'cuda', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

print(f"Setting up the embeddings model...")
embeddings = HuggingFaceEmbeddings(
    model_name=   EMBEDDING_MODEL_NAME,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

Setting up the embeddings model...


In [34]:
ls /projects/RAG2/scotia-2/Datasets-Scotia-2/IBIS

'11114CA Wheat Farming in Canada Industry Report.pdf'
'11115CA Corn Farming in Canada Industry Report.pdf'
'33639CA Auto Parts Manufacturing in Canada Industry Report.pdf'
'44111CA New Car Dealers in Canada Industry Report.pdf'
'48412CA Long-Distance Freight Trucking in Canada Industry Report.pdf'
'48422CA Local Specialized Freight Trucking in Canada Industry Report.pdf'
'48423CA Long-Distance Specialized Freight Trucking in Canada Industry Report.pdf'


In [39]:
directory_path = "/projects/RAG2/scotia-2/Datasets-Scotia-2/IBIS"
file_list = [
   "33639CA Auto Parts Manufacturing in Canada Industry Report.pdf"
]  # Replace with your actual file names

# Load only the specified files
docs = []
for file_name in file_list:
    file_path = os.path.join (directory_path, file_name)
    loader = PyPDFLoader(file_path)
    docs.extend(loader.load())  # Append loaded pages to the list

In [28]:
%%time
ignore
# Load the IBIS pdfs
#directory_path = "./source_documents"
directory_path = "/projects/RAG2/scotia-2/Datasets-Scotia-2/IBIS"
loader = PyPDFDirectoryLoader(directory_path)
docs = loader.load()
print(f"Number of source documents: {len(docs)}")



Number of source documents: 280
CPU times: user 11.5 s, sys: 176 ms, total: 11.6 s
Wall time: 11.8 s


In [20]:
import nltk
from nltk.corpus import words
import re
nltk.download('words')

[nltk_data] Downloading package words to /h/ws_ikharchuk/nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [29]:
def merge_adjacent_words3(word_list):
    english_words = set(words.words())
    i = 0
    while i < len(word_list) - 2:
        combined_word = word_list[i] + word_list[i + 1]+ word_list[i + 1]
        if (combined_word.lower() in english_words) |(combined_word.lower().strip('s').strip('es').strip('ed') in english_words):
            word_list[i] = combined_word
            del word_list[i + 1]
            del word_list[i + 1]
        else:
            i += 1
    return word_list

In [30]:
def merge_adjacent_words2(word_list):
    english_words = set(words.words())
    new_list = []
    i = 0
    while i < len(word_list) - 1:
        combined_word = word_list[i] + word_list[i + 1]
        if (combined_word.lower() in english_words) |(combined_word.lower().strip('s').strip('es').strip('ed') in english_words):
            word_list[i] = combined_word
            del word_list[i + 1]
        else:
            i += 1
    return word_list

In [31]:
def process_string(test):
    test =test.replace('. ', ' ').replace('?', ' ').replace('\n', ' ')
    words_list = test.split()

    words_list = [x.replace('•', ' ')  for x in words_list]

    #words_list =[re.sub(r'[^A-Za-z0-9\s]', '', x) for x in words_list ]
    return ' '.join(merge_adjacent_words2(merge_adjacent_words3(words_list)))
#process_string(test)

In [32]:
# %%time
# for doc in docs[:]:
#     doc.page_content = process_string(doc.page_content)


### Split the documents into smaller chunks

In [40]:
%%time
#Process PDS
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=32)
chunks = text_splitter.split_documents(docs)
print(f"Number of text chunks: {len(chunks)}")

Number of text chunks: 53
CPU times: user 8.13 ms, sys: 3.53 ms, total: 11.7 ms
Wall time: 11.2 ms


In [41]:
# Adding Reuters data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=32)

In [42]:
def read_csv_from_directory(directory_path):
    dataframes= []
    for filename in os.listdir(directory_path):
        if filename.endswith('.csv'): 
            file_path = os.path.join(directory_path, filename)
            df = pd.read_csv(file_path)
            df["source"] = filename
            # print(df.head(1))
            dataframes.append(df)
    return pd.concat(dataframes, ignore_index=True)

### Load cleaned PFD from txt files 

In [50]:
ignore
directory_path  ='/projects/RAG2/scotia-2/Datasets-Scotia-2/PDF_text'

chunks=[]
for filename in os.listdir(directory_path):
    if filename.endswith('.txt'):
        file_path = os.path.join(directory_path, filename)
        print (file_path)
        loader = TextLoader(file_path)

        # Load the document

        document = loader.load()
        chunks2 =text_splitter.split_documents(document)
        print(f"Number of text chunks: {len(chunks2)}")
        chunks= chunks +chunks2

In [44]:
#Load news data
file_paths = [#'/projects/RAG2/scotia-2/Datasets-Scotia-2/Agriculture_txt/agri_ca_co.csv', 
              #'/projects/RAG2/scotia-2/Datasets-Scotia-2/Transport_txt/transport_CA.csv', 
              '/projects/RAG2/scotia-2/Datasets-Scotia-2/Auto_txt/auto_ca.csv', 
             ]
def load_txt_file(file_path):
    # Create a TextLoader instance

    loader = TextLoader(file_path)

    # Load the document

    document = loader.load()
    chunks2 =text_splitter.split_documents(document)
    print(f"Number of text chunks: {len(chunks2)}")
    return chunks2

In [45]:
%%time
for file_path in file_paths:
    chunks2 = load_txt_file(file_path)
    chunks= chunks +chunks2

Number of text chunks: 897
CPU times: user 51.3 ms, sys: 22.9 ms, total: 74.2 ms
Wall time: 77.7 ms


#### Define the embeddings model

## Retrieval: Make the document chunks available via a retriever

The retriever will identify the document chunks that most closely match our original query. (This takes about 1-2 minutes)

In [46]:
%%time
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

CPU times: user 30.2 s, sys: 812 ms, total: 31 s
Wall time: 30.5 s


In [47]:
industries  = ["auto dealer", "wheat farming", 'trucking', 'auto']
#industry  = "wheat farming"
industry ="auto dealer"
def refresh_queries(industry):
    query = f"Where are most {industry} companies are located in Canada?"
    query1  = f' Who are the main players among canadian {industry} companies' # bad

    query2 = f'What is the profit margin of Canadian {industry} companies'

    #query = f'Who are the main palyers in Canadian  {industry}?' # bad
    query3  = f' Which companies are competitors for canadian {industry} internationally?' # bad

    query4 = f'''Please provide a summary of news  grouping the most important event for the {industry} into trends. Is there is anything could be highlighted regional trends happening in Alberta, BC, and Ontatio?How profit margin of the {industry}  companies has changed in 2024. What was the main reasons?'''

    query5 =f''' What is the level of {industry}  consolidation in the Canadian sector, and what are the primary drivers behind this trend? '''

    query6 =f''' What are the primary factors contributing to the supply demand imbalance  in the {industry}  , and what strategies are {industry}  companies employing to address this issue? '''

    query7 = f'''How have fluctuating costs impacted the profitability of Canadian {industry}  companies over the past five years, and what strategies have they employed to mitigate this volatility? '''

    query8 = f'''To what extent has the adoption of new technologies impacted operational efficiency and cost structures within the Canadian {industry}   companies? What are new  technology opportunities in the sector'''

    query9 = f'''How significant is the competition from alternative providers for Canadian {industry}  and how are these companies adapting to this competitive landscape. What is the substitution risk.'''

    query10 = f'''What are the key regulatory and policy challenges facing the Canadian {industry}  companies(e.g., hours of service regulations, environmental regulations, safety standards), and how are these regulations impacting industry operations and profitability?''' 


    query11 = f'''What is the level of government support (subsidies, grants, incentives) available to the Canadian {industry}  , and are any changes expected? '''
    queries  = [query1, query2, query3, query4, query5, query6, query7, query8, query9, query10, query11 ]
    return queries 


In [48]:
headers =  [
    "Key Players ",
    "Profit Margins",
    "International Competitors ",
    "Trends in the Industry: Key Events and Regional Insights",
    "Consolidation in the Sector",
    "Supply and Demand Imbalance  Causes and Solutions",
    "Impact of Cost Fluctuations on Profitability",
    "Technological Advancements in industry",
    "Competitive Pressure",
    "Regulatory and Policy Challenges ",
    "Government Support"
]

In [49]:
industry ="auto parts"
queries = refresh_queries(industry)

In [53]:
# %%time
# retrieved_docs = retriever.invoke(query6)

# pretty_print_docs(retrieved_docs)

In [54]:
#result = rag_pipeline.invoke(input=query6 )

In [55]:
#print (result['result'])

## Now send the query to the RAG pipeline

In [56]:
# %%time
# rag_pipeline = RetrievalQA.from_llm(llm=llm, retriever=retriever)
# query =query6 +' Please answer in 2 sentences maximum. ' #If answer is not available, answer NA 
# result = rag_pipeline.invoke(input=query)
# result = result['result']
# print(f"Result: \n\n{result.replace ('According to the provided context, ', '')}")

# Loop

In [60]:
%%time

def run_questions(queries):
    answers=[]
    for i, query in enumerate (queries):
        query = query + ' Please answer in 5 sentences maximum. ' 
        retrieved_docs = retriever.invoke(query)
        rag_pipeline = RetrievalQA.from_llm(llm=llm, retriever=retriever)
        result = rag_pipeline.invoke(input=query)
        result_txt = result['result']

        print(f"Result_{i+1}: \n\n{result_txt}")

        answers.append(result_txt)
    answers =[x.replace ('According to the provided context, ', '') for x in answers]
    answers  = [headers[i]+ '\n'+ answers[i]+ '\n' for i  in range (len (answers)) ]
    all_answers = '\n'.join(answers)
    
    return all_answers
    
    

CPU times: user 9 µs, sys: 1e+03 ns, total: 10 µs
Wall time: 21 µs


In [57]:
industries  = ["auto dealer", "wheat farming", 'trucking']
#industry  = "wheat farming"
industry ="auto parts"
#industry ="auto parts"
#industry ="wheat farming"
#industry ="trucking"
queries = refresh_queries(industry)

In [58]:
for i, q in enumerate (queries[:3], 1) :
    print (i, q)

1  Who are the main players among canadian auto parts companies
2 What is the profit margin of Canadian auto parts companies
3  Which companies are competitors for canadian auto parts internationally?


In [61]:
%%time
answers =run_questions(queries) 

Result_1: 

I don't have information on the main players among Canadian auto parts companies.
Result_2: 

According to the IBIS World report, the total profit margin for the Canadian auto parts industry has been declining over the years, from 8.7% in 2019-2024 to 6.6%. This decline is attributed to various factors, including climbing interest rates, offshoring of production, and subdued innovation in Canada. The industry's profit margin has been impacted by the trend of aftermarkets, such as repair shops, generating lower per-unit revenue than automobile manufacturers. As a result, auto parts manufacturers have been forced to adapt by moving production offshore, targeting smaller companies with lucrative contracts, and focusing on higher-margin products. Overall, the Canadian auto parts industry's profit margin has been under pressure due to these structural changes.
Result_3: 

I don't know the specific companies that are competitors for Canadian auto parts internationally. The provid

In [64]:
# Stop


In [63]:
with open(f"../Output/by_industry/{industry}_separate.txt", "w", encoding="utf-8") as file:

    file.write(answers)

In [79]:
pwd

'/fs01/home/ws_ikharchuk/rag_bootcamp_ik/document_search'

# Setting up having two vector stores


In [37]:
#PDF 

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1250, chunk_overlap=32)
chunks5 = text_splitter.split_documents(docs)
print(f"Number of text chunks: {len(chunks)}")

Number of text chunks: 4770


In [38]:
%%time
vectorstore_pdf = FAISS.from_documents(chunks5, embeddings)
retriever_pdf = vectorstore_pdf.as_retriever(search_kwargs={"k": 3})

CPU times: user 11.7 s, sys: 31.8 ms, total: 11.7 s
Wall time: 11 s


In [39]:
# News
chunks6=[]

for file_path in file_paths:
    chunks7 = load_txt_file(file_path)
    chunks6= chunks6 +chunks7

Number of text chunks: 1212
Number of text chunks: 927
Number of text chunks: 2065


In [40]:
%%time
# news
vectorstore_news = FAISS.from_documents(chunks6, embeddings)
retriever_news = vectorstore_news.as_retriever(search_kwargs={"k": 3})

CPU times: user 1min 27s, sys: 357 ms, total: 1min 28s
Wall time: 1min 23s


 ## Querying

In [35]:
%%time
retrieved_docs1 = retriever_news.invoke(query4)
print ('from News')
pretty_print_docs(retrieved_docs1)

NameError: name 'retriever_news' is not defined

In [42]:
%%time
retrieved_docs2 = retriever_pdf.invoke(query4)
print ('from PDF')
pretty_print_docs(retrieved_docs2)

from PDF
Document 1:

Whats impac ting Ne wRoads A utomo tive Groups perf ormanc e NewRoads A utomo tive Group pur chased Ne wmark et Honda now called Ne wRoads Honda  NewRoad A utomo tive announc edthe acquisition too chase Ne wRoads too itss too oss York Region Also this isee ted to strengthen the c ompan ys roster of servic esand v ehicl esYou can view and do wnload moree ydede on my ibisworld com Retail Trade In Canada   44 111CA New Car Deal ers in Canada 25 www ibis world com November 202 4
----------------------------------------------------------------------------------------------------
Document 2:

Profit Margin Total profit margin annual change from 20 11  2029 Profit Margin 2pp1pp0pp1pp2pp3pp 2012 2014 2016 2018 2020 2022 202 4 Source IBIS WorldTotal Profit 3 8bn 19240 8 Profit Margin 2 0 19240 2 pp Profit per Business 810 8k Current Performanc e201924 Revenue C AGR 2 8 Whats driving current industry perf ormanc eNew car deal ers have endur edv olatile conditions  The pande

In [43]:
retrieved_docs=retrieved_docs1 +retrieved_docs2

In [44]:
%%time
rag_pipeline = RetrievalQA.from_llm(llm=llm, retriever=retriever)
query =query6 +' Please answer in 2 sentences maximum. If answer is not available, answer NA ' 
result = rag_pipeline.invoke(input=query +' Please answer in 2 sentences maximum. If answer is not available, answer NA ' )
print(f"Result: \n\n{result['result'].replace ('According to the provided context, ', '')}")

Result: 

<think>
Okay, so I need to figure out the answer to the user's question about the innovations in auto dealers. Let me start by reading through the provided context carefully.

First, I see that the user provided several contexts, but the main focus is on the context about auto dealers and their innovations. The user's question is asking for a concise answer in two sentences, and if it's not available, just say NA.

Looking at the context, it mentions that auto dealers are adapting to industry changes, especially with electric vehicles and hybrid powertrains. They are also using social media and online transactions to increase their revenue. Additionally, they are leveraging innovative marketing techniques and partnerships to enhance their products and services.

So, the key points are:
1. Auto dealers are innovating with electric and hybrid vehicles.
2. They are using social media and online transactions to boost revenue.
3. They are employing innovative marketing and partner

In [45]:
result['result'].split('\n')[-1]

'Auto dealers are innovating with electric and hybrid vehicles, leveraging social media and online transactions to boost revenue, and employing innovative marketing and partnerships to enhance their offerings.'

In [46]:
# previous 
print ('Auto dealers are innovating with electric and hybrid vehicles, leveraging social media and online transactions to boost revenue, and employing innovative marketing and partnerships to enhance their offering.')

Auto dealers are innovating with electric and hybrid vehicles, leveraging social media and online transactions to boost revenue, and employing innovative marketing and partnerships to enhance their offering.


In [ ]:
`